### Эксперименты с моделями: удаление дублирующих рамок

1. Модель owlv2

In [1]:
import pandas as pd

from models.metrics import evaluate, iou_matrix
from models.pipeline import read_jsonl, read_manifest, write_jsonl


def remove_duplicate_boxes(boxes, threshold=0.85):
    kept = []
    for box in boxes:
        if not kept or not (iou_matrix(kept, [box]) >= threshold).any():
            kept.append(box)
    return kept


team = read_manifest("data/val_dataset.json")
golden = read_jsonl("data/golden.jsonl")
owl_before = read_jsonl("results/predictions/owlv2.jsonl")

if owl_before.keys() != team.keys() or golden.keys() != team.keys():
    raise ValueError("Списки photo_id в датасете, эталоне и предсказаниях различаются")

owl_after = {
    photo_id: {**row, "boxes": remove_duplicate_boxes(row["boxes"])}
    if not row.get("error") else row
    for photo_id, row in owl_before.items()
}

before_metrics, _ = evaluate(team, golden, owl_before)
after_metrics, _ = evaluate(team, golden, owl_after)

display(pd.DataFrame(
    [before_metrics, after_metrics],
    index=["OWLv2 — исходный", "OWLv2 — без дублей"],
)[["f1_50", "f1_75", "fp_50", "negative_photos_with_fp"]].round(4))

write_jsonl("results/experiments/owlv2__dedup085.jsonl", owl_after.values())

,f1_50,f1_75,fp_50,negative_photos_with_fp
OWLv2 — исходный,0.4760,0.3486,464,103
OWLv2 — без дублей,0.4776,0.3498,461,103


2. dino

In [2]:
dino_before = read_jsonl("results/predictions/grounding_dino.jsonl")

if dino_before.keys() != team.keys():
    raise ValueError("photo_id в предсказаниях Grounding DINO и датасете различаются")

dino_after = {
    photo_id: {**row, "boxes": remove_duplicate_boxes(row["boxes"], threshold=0.85)}
    if not row.get("error") else row
    for photo_id, row in dino_before.items()
}

dino_before_metrics, _ = evaluate(team, golden, dino_before)
dino_after_metrics, _ = evaluate(team, golden, dino_after)

display(pd.DataFrame(
    [dino_before_metrics, dino_after_metrics],
    index=["Grounding DINO — исходный", "Grounding DINO — без дублей"],
)[["f1_50", "f1_75", "fp_50", "negative_photos_with_fp"]].round(4))

write_jsonl("results/experiments/grounding_dino__dedup085.jsonl", dino_after.values())

,f1_50,f1_75,fp_50,negative_photos_with_fp
Grounding DINO — исходный,0.3788,0.3510,655,182
Grounding DINO — без дублей,0.5854,0.5366,275,182


3. qwen запускался на гпу в kaggle: [notebook](./qwen_experiment.ipynb)

4. Сбор результатов

In [8]:
import sys
from decimal import Decimal
from pathlib import Path

import pandas as pd
from IPython.display import display

# Находим корень репозитория перед импортом локальных модулей.
cwd = Path.cwd().resolve()
ROOT = next(
    (
        path for path in (cwd, *cwd.parents)
        if (path / "models" / "metrics.py").is_file()
        and (path / "data" / "val_dataset.json").is_file()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Запусти ноутбук из папки репозитория")

sys.path.insert(0, str(ROOT))

from models.metrics import evaluate
from models.pipeline import read_jsonl, read_manifest

DATA = ROOT / "data"
RESULTS = ROOT / "results"
PREDICTIONS = RESULTS / "predictions"
EXPERIMENTS = RESULTS / "experiments"

team = read_manifest(DATA / "val_dataset.json")
golden = read_jsonl(DATA / "golden.jsonl")

if len(team) != 400 or golden.keys() != team.keys():
    raise ValueError("В датасете и эталоне должны совпадать все 400 photo_id")

baseline_paths = {
    model: PREDICTIONS / f"{model}.jsonl"
    for model in ("grounding_dino", "owlv2", "qwen")
}

# Для Qwen берём исходные предсказания из архива самого эксперимента.
experiments = [
    (
        "grounding_dino",
        "Удаление дублей",
        baseline_paths["grounding_dino"],
        EXPERIMENTS / "grounding_dino__dedup085.jsonl",
    ),
    (
        "owlv2",
        "Удаление дублей",
        baseline_paths["owlv2"],
        EXPERIMENTS / "owlv2__dedup085.jsonl",
    ),
    (
        "qwen",
        "Новый промпт",
        EXPERIMENTS / "qwen__reference.jsonl",
        EXPERIMENTS / "qwen__strict_negative_prompt.jsonl",
    ),
]

# Читаем каждый файл один раз и считаем метрики на общем эталоне.
paths = dict.fromkeys([
    *baseline_paths.values(),
    *(path for _, _, before, after in experiments for path in (before, after)),
])

predictions = {}
scores = {}

for path in paths:
    if not path.is_file():
        raise FileNotFoundError(f"Не найден файл с предсказаниями: {path}")

    rows = read_jsonl(path)
    if rows.keys() != team.keys():
        raise ValueError(f"{path.name}: набор photo_id отличается от датасета")

    predictions[path] = rows
    scores[path], _ = evaluate(team, golden, rows)

    # Формула из PDF. Используем F1@0.50 до округления для вывода.
    f1 = Decimal(str(scores[path]["f1_50"]))
    scores[path]["score_10"] = int(min(10, 10 * (f1 + Decimal("0.3"))))

# Общий лидерборд начинается с трёх бейзлайнов.
leaderboard_rows = [
    {"run": path.stem, "model": model, "variant": "Бейзлайн", **scores[path]}
    for model, path in baseline_paths.items()
]

# Для каждого эксперимента показываем метрики до, после и разницу.
comparison_rows = []
metrics = (
    "f1_50", "f1_75", "precision_50", "recall_50", "fp_50",
    "negative_photos_with_fp", "technical_errors", "score_10",
)

for model, experiment, before, after in experiments:
    comparison_row = {"model": model, "experiment": experiment}

    for metric in metrics:
        before_value = scores[before][metric]
        after_value = scores[after][metric]
        comparison_row[f"{metric}_before"] = before_value
        comparison_row[f"{metric}_after"] = after_value
        comparison_row[f"{metric}_delta"] = after_value - before_value

    comparison_rows.append(comparison_row)

    # Выводим референс отдельной строкой, если его предсказания
    # отличаются от бейзлайна этой модели.
    if predictions[before] != predictions[baseline_paths[model]]:
        leaderboard_rows.append({
            "run": before.stem,
            "model": model,
            "variant": "Референс эксперимента",
            **scores[before],
        })

    leaderboard_rows.append({
        "run": after.stem,
        "model": model,
        "variant": experiment,
        **scores[after],
    })

comparison = pd.DataFrame(comparison_rows)
leaderboard = (
    pd.DataFrame(leaderboard_rows)
    .sort_values("f1_50", ascending=False, kind="stable")
    .reset_index(drop=True)
)

comparison.to_csv(RESULTS / "experiments_comparison.csv", index=False)
leaderboard.to_csv(RESULTS / "leaderboard.csv", index=False)

display(comparison[[
    "model", "experiment",
    "f1_50_before", "f1_50_after", "f1_50_delta",
    "f1_75_before", "f1_75_after",
    "score_10_before", "score_10_after", "score_10_delta",
]].round(4))

display(leaderboard[[
    "run", "variant", "f1_50", "f1_75", "precision_50", "recall_50",
    "negative_photos_with_fp", "technical_errors", "score_10",
]].round(4))

,model,experiment,f1_50_before,f1_50_after,f1_50_delta,f1_75_before,f1_75_after,score_10_before,score_10_after,score_10_delta
0,grounding_dino,Удаление дублей,0.3788,0.5854,0.2065,0.3510,0.5366,6,8,2
1,owlv2,Удаление дублей,0.4760,0.4776,0.0016,0.3486,0.3498,7,7,0
2,qwen,Новый промпт,0.6508,0.8742,0.2234,0.5841,0.7889,9,10,1


,run,variant,f1_50,f1_75,precision_50,recall_50,negative_photos_with_fp,technical_errors,score_10
0,qwen__strict_negative_prompt,Новый промпт,0.8742,0.7889,0.8167,0.9404,43,0,10
1,qwen,Бейзлайн,0.6508,0.5841,0.4976,0.9404,202,0,9
2,grounding_dino__dedup085,Удаление дублей,0.5854,0.5366,0.4259,0.9358,182,0,8
3,owlv2__dedup085,Удаление дублей,0.4776,0.3498,0.3160,0.9771,103,0,7
4,owlv2,Бейзлайн,0.4760,0.3486,0.3146,0.9771,103,0,7
5,grounding_dino,Бейзлайн,0.3788,0.3510,0.2375,0.9358,182,0,6
